In [ ]:
#@title 1. Load source code
SOURCE = "github" #@param ["github", "drive archive"]
SOURCE_ARCHIVE = "/content/drive/MyDrive/humanoid/humanoid-source.tar.gz" #@param {type:"string"}
REPO = "university-of-toronto-humanoid-robotics/humanoid" #@param {type:"string"}
BRANCH = "kartik/rpl" #@param {type:"string"}

from pathlib import Path
import os
import shutil
import subprocess
import sys
from IPython.display import HTML, display

REPO_DIR = Path("/content/humanoid")
PYTHON = "/content/mjlab-venv/bin/python"
if SOURCE == 'github':
    if shutil.which("gh") is None:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "gh"], check=True)
    git_env = {**os.environ, "GH_BROWSER": "echo", "GH_PROMPT_DISABLED": "1", "GIT_TERMINAL_PROMPT": "0"}
    git_env.pop("GH_TOKEN", None)
    git_env.pop("GITHUB_TOKEN", None)
    if subprocess.run(["gh", "auth", "status", "--hostname", "github.com"], env=git_env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode:
        display(HTML('<a href="https://github.com/login/device" target="_blank">Open GitHub authorization</a><p>Enter the code below and approve access. This cell waits for you.</p>'))
        login = subprocess.Popen(["gh", "auth", "login", "--web", "--hostname", "github.com", "--git-protocol", "https"], env=git_env, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        try:
            for line in login.stdout:
                print(line, end="", flush=True)
            if login.wait():
                raise RuntimeError("GitHub login failed. Check the output above.")
        except KeyboardInterrupt:
            login.send_signal(2)
            login.wait()
            raise
    subprocess.run(["gh", "auth", "setup-git", "--hostname", "github.com"], env=git_env, check=True)
    url = f"https://github.com/{REPO}.git"
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, url, str(REPO_DIR)], env=git_env, check=True)
    else:
        origin = subprocess.check_output(["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True).strip()
        assert origin.lower().removesuffix(".git") == url.lower().removesuffix(".git"), "Existing checkout uses a different repository; use a fresh runtime."
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], env=git_env, check=True)
        current = subprocess.check_output(["git", "-C", str(REPO_DIR), "branch", "--show-current"], text=True).strip()
        if current != BRANCH:
            subprocess.run(["git", "-C", str(REPO_DIR), "switch", BRANCH], env=git_env, check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], env=git_env, check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "submodule", "update", "--init", "--recursive", "training/unitree_rl_mjlab"], env=git_env, check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"], check=True)
elif SOURCE == 'drive archive':
    from google.colab import drive
    drive.mount('/content/drive')
    archive = Path(SOURCE_ARCHIVE)
    assert archive.is_file(), f'Upload the source archive first: {archive}'
    entries = subprocess.check_output(['tar', '-tzf', str(archive)], text=True).splitlines()
    assert entries and all(
        entry == 'humanoid' or (entry.startswith('humanoid/') and '..' not in Path(entry).parts)
        for entry in entries
    ), 'Archive must contain a single top-level humanoid/ folder.'
    assert any(entry.startswith('humanoid/training/unitree_rl_mjlab/src/') for entry in entries), 'Archive is missing the Unitree submodule.'
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(['tar', '-xzf', str(archive), '-C', str(REPO_DIR.parent)], check=True)
    assert (REPO_DIR / 'training/scripts/train_student_bc.py').is_file(), 'Archive predates the student trainer; recreate it from the latest local code.'
    print('Loaded source archive:', archive)
else:
    raise ValueError(f'Unknown source: {SOURCE}')


In [ ]:
# 2. Install in an isolated venv using the same pins as the team's Docker setup.
# The requirements filename says native, but pip selects Linux/CUDA wheels here.
import time
assert (3, 10) <= sys.version_info[:2] < (3, 14), "Use Python 3.10–3.13 with these pins."
# Colab's Python version may not have a matching apt venv package.
venv_dir = Path(PYTHON).parent.parent
venv_ready = Path(PYTHON).is_file() and subprocess.run([PYTHON, "-m", "pip", "--version"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
if not venv_ready:
    print("Creating the Python environment...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run([sys.executable, "-m", "uv", "venv", "--python", sys.executable, "--seed", "--clear", str(venv_dir)], check=True)
# RSL-RL passes start_method to W&B; W&B 0.29 removed that setting.
print("Installing training dependencies; pip output appears below.", flush=True)
install = subprocess.Popen(
    [PYTHON, "-u", "-m", "pip", "install", "-v", "--progress-bar", "raw",
     "-r", str(REPO_DIR / "docs/requirements-native-mjlab.txt"), "wandb<0.29"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
started = time.monotonic()
try:
    for line in install.stdout:
        print(line, end="", flush=True)
    return_code = install.wait()
except KeyboardInterrupt:
    install.terminate()
    install.wait()
    raise
if return_code:
    raise subprocess.CalledProcessError(return_code, install.args)
print(f"Dependencies installed in {time.monotonic() - started:.0f}s.", flush=True)


In [ ]:
# 3. Mount Drive and verify the GPU.
from google.colab import drive
drive.mount('/content/drive')
WORK_DIR = Path('/content/drive/MyDrive/humanoid')
WORK_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run(['nvidia-smi'], check=True)
subprocess.run([PYTHON, '-c', "import torch, warp as wp; wp.init(); assert torch.cuda.is_available() and wp.is_cuda_available(); print(torch.cuda.get_device_name(0)); wp.zeros(1, device='cuda:0'); wp.synchronize_device('cuda:0')"], check=True)
print('Student checkpoints:', WORK_DIR / 'logs/student_distill_v2')


In [ ]:
# 4. Choose the six-expert workflow, experts, and separate BC and DAgger settings.
# Rerun this cell to discover newly saved checkpoints on Drive.
import ipywidgets as widgets
from IPython.display import display
FEATURES = ('ascend', 'descend', 'ramps', 'obstacles', 'maze', 'flat')
assert (REPO_DIR / 'training/g1_comp/rpl.py').is_file(), 'Source archive is missing the RPL tasks.'
registry = subprocess.check_output([PYTHON, '-c', "import _setup; from mjlab.tasks.registry import list_tasks; print('RPL-Maze' in list_tasks())"], cwd=REPO_DIR / 'training/scripts', text=True)
assert registry.strip().splitlines()[-1] == 'True', 'Selected branch does not contain RPL-Maze; push the new task or recreate the Drive archive.'
DEFAULT_EXPERTS = {
    'ascend': 'model_v5_1998_best.pt', 'descend': 'model_v2_3496_best.pt',
    'ramps': 'model_v2_3498_best.pt', 'obstacles': 'model_1998.pt',
    'maze': 'model_11991.pt', 'flat': 'model_499.pt',
}
MODEL_DIRS = (REPO_DIR / 'training/models/RPL', WORK_DIR / 'models/RPL')

def checkpoint_options(paths):
    files = sorted({p.resolve() for p in paths if p.is_file()}, key=lambda p: p.stat().st_mtime, reverse=True)
    return [(str(p.relative_to(REPO_DIR)) if p.is_relative_to(REPO_DIR) else str(p), str(p)) for p in files]

expert_widgets = {}
for feature in FEATURES:
    paths = [p for root in MODEL_DIRS for p in (root / feature).glob('*.pt')]
    if feature == 'maze':
        paths += list((REPO_DIR / 'training/models/old/g1_maze_finetuned').glob('*.pt'))
        paths += list((WORK_DIR / 'logs/rsl_rl/rpl_maze').glob('*/model_*.pt'))
    elif feature == 'flat':
        paths += list((REPO_DIR / 'training/models/RPL/flat_baseline').glob('*.pt'))
        paths += list((WORK_DIR / 'logs/rsl_rl/g1_flat_heightmap').glob('*/model_*.pt'))
    choices = checkpoint_options(paths)
    if not choices:
        choices = [(f'Choose a {feature} checkpoint after training', '')]
    default = next((value for _, value in choices if Path(value).name == DEFAULT_EXPERTS[feature]), choices[0][1])
    expert_widgets[feature] = widgets.Dropdown(options=choices, value=default, description=feature.title())

bc_files = list((WORK_DIR / 'logs/student_distill_v2/bc').glob('*/checkpoints/student_*.pt'))
bc_files += list((REPO_DIR / 'training/models/RPL/student_v2').glob('**/student_*.pt'))
bc_choices = checkpoint_options(bc_files) or [('No BC checkpoint found', '')]
demo_files = list((WORK_DIR / 'logs/student_distill_v2/bc').glob('*/checkpoints/demos_*.pt'))
demo_files += list((REPO_DIR / 'training/models/RPL/student_v2').glob('**/demos_*.pt'))
demo_choices = checkpoint_options(demo_files + bc_files) or [('No BC demos found', '')]
default_demo = Path(bc_choices[0][1]).with_name(Path(bc_choices[0][1]).name.replace('student_', 'demos_', 1)) if bc_choices[0][1] else None
default_demo = str(default_demo) if default_demo and default_demo.is_file() else demo_choices[0][1]
resume_files = list((WORK_DIR / 'logs/student_distill_v2').glob('*/*/checkpoints/student_*.pt'))
resume_choices = checkpoint_options(resume_files) or [('No saved run found', '')]
mode_widget = widgets.Dropdown(options=[
    ('Joint: BC then DAgger', 'joint'), ('BC only', 'bc'),
    ('DAgger: fresh student', 'scratch'), ('DAgger: BC weights only', 'bc-weights'),
    ('DAgger: BC weights + demos', 'bc-with-demos'),
    ('Resume BC', 'resume-bc'), ('Resume DAgger', 'resume-dagger'),
], value='joint', description='Workflow')
weight_widget = widgets.Dropdown(options=bc_choices, description='BC weights', layout=widgets.Layout(width='95%'))
demo_widget = widgets.Dropdown(options=demo_choices, value=default_demo, description='BC demos', layout=widgets.Layout(width='95%'))
resume_widget = widgets.Dropdown(options=resume_choices, description='Resume', layout=widgets.Layout(width='95%'))

def stage_settings(prefix, defaults):
    return {
        'num_envs': widgets.IntText(value=defaults[0], description=f'{prefix} envs'),
        'updates': widgets.IntText(value=defaults[1], description=f'{prefix} updates'),
        'rollout_steps': widgets.IntText(value=defaults[2], description='Rollout steps'),
        'batch_size': widgets.IntText(value=4096, description='Batch size'),
        'gradient_steps': widgets.IntText(value=4, description='Gradient steps'),
        'replay_capacity': widgets.IntText(value=20000, description='Replay cap'),
        'save_every': widgets.IntText(value=200, description='Save every'),
        'learning_rate': widgets.FloatText(value=3e-4, description='Learning rate'),
    }

bc_widgets = stage_settings('BC', (1020, 200, 24))
dagger_widgets = stage_settings('DAgger', (1020, 1000, 24))
run_widget = widgets.Text(value='student_v2_1', description='Run name', continuous_update=False)
logger_widget = widgets.Dropdown(options=['wandb', 'tensorboard', 'none'], value='wandb', description='Logger')
project_widget = widgets.Text(value='humanoid-student', description='W&B project', continuous_update=False)
entity_widget = widgets.Text(value='', description='W&B entity', continuous_update=False)

def show_sources(*_):
    mode = mode_widget.value
    weight_widget.layout.display = '' if mode in ('bc-weights', 'bc-with-demos') else 'none'
    demo_widget.layout.display = '' if mode == 'bc-with-demos' else 'none'
    resume_widget.layout.display = '' if mode.startswith('resume-') else 'none'
mode_widget.observe(show_sources, names='value')
show_sources()
display(widgets.VBox([
    mode_widget,
    widgets.HTML('<b>Six experts and logging</b> — flat uses random velocity and heading commands; maze follows its doorway route.'),
    widgets.GridBox(list(expert_widgets.values()), layout=widgets.Layout(grid_template_columns='repeat(2, 1fr)')),
    weight_widget, demo_widget, resume_widget,
    widgets.GridBox([run_widget, logger_widget, project_widget, entity_widget],
                    layout=widgets.Layout(grid_template_columns='repeat(2, 1fr)')),
    widgets.HTML('<b>BC settings</b> — used for BC only, joint, or resume BC'),
    widgets.GridBox(list(bc_widgets.values()), layout=widgets.Layout(grid_template_columns='repeat(4, 1fr)')),
    widgets.HTML('<b>DAgger settings</b> — used for DAgger only, joint, or resume DAgger'),
    widgets.GridBox(list(dagger_widgets.values()), layout=widgets.Layout(grid_template_columns='repeat(4, 1fr)')),
]))


In [ ]:
# 5. Train the selected stage(s). Joint mode starts DAgger only after BC succeeds.
from datetime import datetime, timezone
from google.colab import userdata

mode = mode_widget.value
run_name = run_widget.value.strip()
assert run_name and '/' not in run_name and '\\' not in run_name, 'Enter a simple run name.'
experts = {}
for feature in FEATURES:
    path = expert_widgets[feature].value
    assert path and Path(path).is_file(), f'Select a {feature} expert.'
    experts[feature] = path

def selected_settings(stage):
    chosen = bc_widgets if stage == 'bc' else dagger_widgets
    values = {name: widget.value for name, widget in chosen.items()}
    assert values['num_envs'] % 6 == 0 and values['num_envs'] >= (60 if stage == 'bc' else 6), f'Invalid {stage} env count.'
    assert all(value > 0 for value in values.values()), f'All {stage} settings must be positive.'
    return values

def training_command(stage, settings, name, *, run_dir=None, resume=None, init_mode=None, weights=None, demos=None):
    script = 'train_student_bc.py' if stage == 'bc' else 'train_student_dagger.py'
    command = [PYTHON, '-u', str(REPO_DIR / 'training/scripts' / script), '--profile', 'six']
    for key, value in settings.items():
        command += ['--' + key.replace('_', '-'), str(value)]
    command += ['--device', 'cuda:0', '--run-name', name,
                '--logger', logger_widget.value, '--wandb-project', project_widget.value.strip()]
    for feature, path in experts.items():
        command += [f'--teacher-{feature}', path]
    if run_dir is not None:
        command += ['--run-dir', str(run_dir)]
    if resume is not None:
        command += ['--resume', str(resume)]
    if init_mode is not None:
        command += ['--init-mode', init_mode]
    if weights is not None:
        command += ['--bc-checkpoint', str(weights)]
    if demos is not None:
        command += ['--bc-demo-checkpoint', str(demos)]
    return command

train_env = os.environ.copy()
if logger_widget.value == 'wandb':
    token = userdata.get('WANDB_TOKEN')
    assert token, 'Add WANDB_TOKEN to Colab secrets or choose another logger.'
    assert project_widget.value.strip(), 'Enter a W&B project name.'
    train_env.update(WANDB_API_KEY=token, WANDB_MODE='online')
    if entity_widget.value.strip():
        train_env['WANDB_ENTITY'] = entity_widget.value.strip()

def run_training(label, command, output_dir):
    print(f'\nStarting {label}; output: {output_dir}', flush=True)
    process = subprocess.Popen(command, cwd=REPO_DIR / 'training', env=train_env,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        return_code = process.wait()
    except KeyboardInterrupt:
        process.send_signal(2)
        process.wait()
        raise
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

base_dir = WORK_DIR / 'logs/student_distill_v2'
if mode == 'joint':
    bc = selected_settings('bc')
    dagger = selected_settings('dagger')
    stamp = datetime.now(timezone.utc).strftime('%Y-%m-%d_%H-%M-%S')
    bc_name, dagger_name = run_name + '_bc', run_name + '_dagger'
    bc_dir = base_dir / 'bc' / (stamp + '_' + bc_name)
    dagger_dir = base_dir / 'dagger' / (stamp + '_' + dagger_name)
    assert not bc_dir.exists() and not dagger_dir.exists(), 'Run directory already exists.'
    bc_command = training_command('bc', bc, bc_name, run_dir=bc_dir)
    bc_weights = bc_dir / 'checkpoints' / f"student_{bc['updates']}.pt"
    bc_demos = bc_dir / 'checkpoints' / f"demos_{bc['updates']}.pt"
    dagger_command = training_command('dagger', dagger, dagger_name, run_dir=dagger_dir,
                                      init_mode='bc-with-demos', weights=bc_weights, demos=bc_demos)
    run_training('BC', bc_command, bc_dir)
    assert bc_weights.is_file() and bc_demos.is_file(), 'BC finished without its final weights and demos files.'
    run_training('DAgger', dagger_command, dagger_dir)
elif mode.startswith('resume-'):
    stage = mode.removeprefix('resume-')
    settings = selected_settings(stage)
    selected = resume_widget.value
    assert selected and Path(selected).is_file(), 'Select a saved checkpoint in cell 4.'
    assert f'/student_distill_v2/{stage}/' in selected, 'Resume mode must match the checkpoint stage.'
    command = training_command(stage, settings, run_name, resume=selected)
    run_training('Resume ' + stage, command, Path(selected).parent.parent)
else:
    stage = 'bc' if mode == 'bc' else 'dagger'
    settings = selected_settings(stage)
    run_dir = base_dir / stage / (datetime.now(timezone.utc).strftime('%Y-%m-%d_%H-%M-%S') + '_' + run_name)
    assert not run_dir.exists(), f'Run directory already exists: {run_dir}'
    weights = demos = None
    if mode in ('bc-weights', 'bc-with-demos'):
        weights = weight_widget.value
        assert weights and Path(weights).is_file(), 'Select a BC weight checkpoint in cell 4.'
    if mode == 'bc-with-demos':
        demos = demo_widget.value
        assert demos and Path(demos).is_file(), 'Select a BC demo file in cell 4.'
    command = training_command(stage, settings, run_name, run_dir=run_dir,
                               init_mode=mode if stage == 'dagger' else None, weights=weights, demos=demos)
    run_training(stage.upper(), command, run_dir)


In [ ]:
# 6. Recent saved student checkpoints. Rerun cell 4 to refresh its selectors.
checkpoints = sorted((WORK_DIR / 'logs/student_distill_v2').glob('*/*/checkpoints/student_*.pt'),
                     key=lambda p: p.stat().st_mtime, reverse=True)
for checkpoint in checkpoints[:15]:
    demos = checkpoint.with_name(checkpoint.name.replace('student_', 'demos_', 1))
    print(checkpoint.relative_to(WORK_DIR), f'({checkpoint.stat().st_size / 1e6:.1f} MB)')
    if demos.is_file():
        print('  demos:', demos.relative_to(WORK_DIR), f'({demos.stat().st_size / 1e6:.1f} MB)')
